In [ ]:
import pygmsh

R = 200e3
δx = 5e3

geometry = pygmsh.built_in.Geometry()

x1 = geometry.add_point([-R, 0, 0], lcar=δx)
x2 = geometry.add_point([+R, 0, 0], lcar=δx)

center1 = geometry.add_point([0, 0, 0,], lcar=δx)
center2 = geometry.add_point([0, -4 * R, 0], lcar=δx)

arcs = [
    geometry.add_circle_arc(x1, center1, x2),
    geometry.add_circle_arc(x2, center2, x1)
]

outer_loop = geometry.add_line_loop(arcs)

x0, y0 = R / 3, -R / 4
ctrl_points1 = [
    geometry.add_point([x0 + R / 5, y0, 0], lcar=δx),
    geometry.add_point([x0, y0 + R / 30, 0], lcar=δx),
    geometry.add_point([x0 - R / 5, y0, 0], lcar=δx),
]
ctrl_points2 = [
    ctrl_points1[-1],
    geometry.add_point([x0, y0 - R / 30, 0], lcar=δx),
    ctrl_points1[0],
]

splines = [
    geometry.add_spline(ctrl_points1),
    geometry.add_spline(ctrl_points2),
]

inner_loop = geometry.add_line_loop(splines)

plane_surface = geometry.add_plane_surface(outer_loop, [inner_loop])

physical_lines = (
    [geometry.add_physical(arc) for arc in arcs]
    + [geometry.add_physical(s) for s in splines]
)
physical_surface = geometry.add_physical(plane_surface)

In [ ]:
import subprocess

with open("ice-shelf.geo", "w") as geo_file:
    geo_file.write(geometry.get_code())
    
subprocess.run("gmsh -2 -format msh2 -o ice-shelf.msh ice-shelf.geo".split(" "))

In [ ]:
import firedrake

mesh = firedrake.Mesh("ice-shelf.msh")

In [ ]:
import icepack.plot
import matplotlib.pyplot as plt

fig, axes = icepack.plot.subplots()
icepack.plot.triplot(mesh, axes=axes)
axes.legend();

In [ ]:
import numpy as np
from numpy import pi as π

inlet_angles = π * np.array([-3/4, -1/2, -1/3, -1/6])
inlet_widths = π * np.array([1/8, 1/12, 1/24, 1/12])

In [ ]:
from firedrake import inner, as_vector

x = firedrake.SpatialCoordinate(mesh)

u_in = 300
h_in = 350
hb = 100
dh, du = 400, 250

hs, us = [], []
for θ, ϕ in zip(inlet_angles, inlet_widths):
    x0 = R * as_vector((np.cos(θ), np.sin(θ)))
    v = -as_vector((np.cos(θ), np.sin(θ)))
    L = inner(x - x0, v)
    W = x - x0 - L * v
    Rn = 2 * ϕ / π * R
    q = firedrake.max_value(1 - (W / Rn)**2, 0)
    hs.append(hb + q * ((h_in - hb) - dh * L /R))
    us.append(firedrake.exp(-4 * (W/R)**2) * (u_in + du * L / R) * v)

In [ ]:
h_expr = firedrake.Constant(hb)
for h in hs:
    h_expr = firedrake.max_value(h, h_expr)
    
u_expr = sum(us)

In [ ]:
Q = firedrake.FunctionSpace(mesh, "CG", 2)
V = firedrake.VectorFunctionSpace(mesh, "CG", 2)

h0 = firedrake.interpolate(h_expr, Q)
u0 = firedrake.interpolate(u_expr, V)

In [ ]:
fig, axes = icepack.plot.subplots()
axes.set_title("Thickness")
colors = icepack.plot.tripcolor(h0, axes=axes)
fig.colorbar(colors);

In [ ]:
fig, axes = icepack.plot.subplots()
axes.set_title("Velocity")
streamlines = icepack.plot.streamplot(
    u0, precision=1e3, density=2e3, axes=axes
)
fig.colorbar(streamlines);

### Modeling

In [ ]:
import icepack

model = icepack.models.IceShelf()

In [ ]:
solver = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])

In [ ]:
T = firedrake.Constant(255.15)
A = icepack.rate_factor(T)

In [ ]:
h = h0.copy(deepcopy=True)
u = solver.diagnostic_solve(
    velocity=u0,
    thickness=h,
    fluidity=A,
)

In [ ]:
fig, axes = icepack.plot.subplots()
streamlines = icepack.plot.streamplot(
    u, precision=1e3, density=2e3, axes=axes
)
fig.colorbar(streamlines, label="meters/year");

In [ ]:
import tqdm

final_time = 400.
num_timesteps = 200
dt = final_time / num_timesteps
a = firedrake.Constant(0.0)

for step in tqdm.trange(num_timesteps):
    h = solver.prognostic_solve(
        dt,
        thickness=h, 
        velocity=u,
        accumulation=a,
        thickness_inflow=h0,
    )
    
    u = solver.diagnostic_solve(
        velocity=u,
        thickness=h,
        fluidity=A,
    )

In [ ]:
fig, axes = icepack.plot.subplots()
colors = icepack.plot.tripcolor(h, axes=axes)
fig.colorbar(colors, label="meters");

In [ ]:
fig, axes = icepack.plot.subplots()
streamlines = icepack.plot.streamplot(
    u, precision=1e3, density=2e3, axes=axes
)
fig.colorbar(streamlines, label="meters/year");

In [ ]:
from firedrake import ds

ν = firedrake.FacetNormal(mesh)
flux = h * inner(u, ν) * ds((2, 3, 4))

In [ ]:
print(f"Flux: {firedrake.assemble(flux) / 1e9} km^3 / year")

In [ ]:
influx = -h * inner(u, ν) * ds(1)
print(f"Influx: {firedrake.assemble(influx) / 1e9} km^3 / year")